💡 **Environment:** `clamp-analyses`

# Description

**Pooled early-enrichment metrics (BEDROC, EF, precision@k).** NB10/11 and `../signif_test` report **AUROC/AUPRC**, which integrate over the *whole* ranked list. Drug repurposing is a **top-of-the-list** problem: you only act on the handful of candidates you would take into validation. This notebook scores the **pooled global** ranking — all 685 (drug, disease) pairs ranked together — with metrics that weight the *top* of the list:

- **BEDROC** (Truchon & Bajorath 2007), the headline — a single number in [0, 1] with α=20, exponentially weighting the top ~1/α = 5% (~34 pairs).
- **Enrichment factor** EF@{1, 5, 10%} — precision in the top fraction over the base rate.
- **precision@k** at k ∈ {10, 20, 50} — fraction of the top-k that are true treatments.

It is a **read-only consumer** of `../signif_test/predictions_paired.pkl` (the aligned 685-pair × 4-method frame; same pooled list `../signif_test` scores with AUROC).

**Load-bearing caveat (high prevalence → low headroom).** After the join the pooled set is **531 pos / 154 neg = 77.5% positive** — the *inverse* of the rare-positive regime these metrics were built for. So random-baseline **BEDROC = base rate ≈ 0.775** (not 0.5), **EF is capped at 1/0.775 ≈ 1.29**, and precision@k's null is 0.775. Absolute values sit near a ceiling for every method; the interpretable quantity is the **paired difference** between methods at the top (computed in `01`).

`rdkit` is **not** installed in `clamp-analyses`, so BEDROC/EF/precision@k are hand-rolled in numpy (verified: random≈0.775, perfect=1.0, worst=0.0). Output: `early_enrichment_observed.csv` (one row per `(metric, method)`), consumed by `01_early_enrichment_summary.ipynb`.

# Modules loading

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd

from pyprojroot import here

# Settings

In [3]:
# The 4 methods, fixed order (matches ../signif_test). gene_based is the
# single-gene baseline (NB06); the three module_based_* are the LV models.
METHOD_ORDER = [
    'gene_based',
    'module_based_archs4',
    'module_based_gtex',
    'module_based_recount2',
]

# Early-enrichment configuration. BEDROC alpha=20 concentrates ~80% of its weight
# on the top ~1/alpha = 5% of the ranked list (~34 of 685 pairs). EF/precision are
# evaluated at small top cuts.
BEDROC_ALPHA = 20.0
EF_CHIS = [0.01, 0.05, 0.10]   # top-fraction cuts for the enrichment factor
PK_KS = [10, 20, 50]           # top-k cuts for precision@k

# Display order for the 7 metrics.
METRIC_ORDER = ['BEDROC', 'EF@1%', 'EF@5%', 'EF@10%', 'P@10', 'P@20', 'P@50']

INPUT_PKL = here(
    'output/03_model_biology/00_archs4/02_drug_disease_associations/'
    'signif_test/predictions_paired.pkl')

OUTPUT_DIR = here(
    'output/03_model_biology/00_archs4/02_drug_disease_associations/early_enrichment_test')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
display(INPUT_PKL)
display(OUTPUT_DIR)

PosixPath('/home/miltondp/projects/clamp/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/signif_test/predictions_paired.pkl')

PosixPath('/home/miltondp/projects/clamp/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/early_enrichment_test')

# Metric helpers

Hand-rolled because `rdkit` is not in the `clamp-analyses` env. All three rank by **descending score** with a **stable** sort, so ties break deterministically by input (pivot) order — fully reproducible. `bedroc_score` returns NaN on a single-class input (undefined); `enrichment_factor` uses `K = ceil(chi*N)` so χ=1% always evaluates a non-empty cut (K = 7 / 35 / 69 at N=685).

In [4]:
def bedroc_score(y_true, scores, alpha=20.0):
    """BEDROC (Truchon & Bajorath 2007). Higher score => more likely positive.
    Returns a value in [0, 1]: random ranking -> base rate (Ra), perfect -> 1,
    worst -> 0. Ties broken deterministically by input order (stable sort)."""
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores, dtype=float)
    N = len(y_true)
    n = int(y_true.sum())
    if n == 0 or n == N:            # single class -> undefined (handled by the bootstrap skip)
        return np.nan
    order = np.argsort(-scores, kind='stable')
    active_ranks = np.where(y_true[order] == 1)[0] + 1   # 1-indexed ranks of positives
    Ra = n / N
    s = np.sum(np.exp(-alpha * active_ranks / N))
    rie_denom = (1.0 / N) * (1 - np.exp(-alpha)) / (np.exp(alpha / N) - 1)
    rie = (s / n) / rie_denom
    return (rie * Ra * np.sinh(alpha / 2)
            / (np.cosh(alpha / 2) - np.cosh(alpha / 2 - alpha * Ra))
            + 1.0 / (1 - np.exp(alpha * (1 - Ra))))


def precision_at_k(y_true, scores, k):
    """Fraction of the top-k ranked items that are positive. Null = base rate."""
    order = np.argsort(-scores, kind='stable')
    return float(np.asarray(y_true).astype(int)[order[:k]].mean())


def enrichment_factor(y_true, scores, chi):
    """Precision in the top ceil(chi*N), divided by the base rate.
    Null = 1; ceiling = 1/base_rate. K via ceil (7/35/69 at N=685)."""
    y_true = np.asarray(y_true).astype(int)
    N = len(y_true)
    n = int(y_true.sum())
    K = int(np.ceil(chi * N))
    order = np.argsort(-scores, kind='stable')
    return (int(y_true[order[:K]].sum()) / K) / (n / N)

# Load paired predictions

The aligned long frame from `../signif_test`: `[trait, drug, method, score, true_class]`. We assert the (trait, drug) pair universe and labels are identical across the four methods (the shared-universe guarantee), then pivot to a wide score matrix with a single shared `y_true`.

In [5]:
predictions = pd.read_pickle(INPUT_PKL)
display(predictions.shape)
display(predictions.head())

# Coverage / pairing check: every method must cover the IDENTICAL (trait, drug)
# set with consistent labels -- the shared-universe guarantee from ../signif_test.
assert not predictions.isna().any().any(), 'unexpected NaNs in predictions_paired.pkl'
missing = [m for m in METHOD_ORDER if m not in set(predictions['method'].unique())]
assert not missing, f'methods absent: {missing}'

pair_sets = {
    m: set(map(tuple, g[['trait', 'drug']].itertuples(index=False, name=None)))
    for m, g in predictions.groupby('method', observed=True)
}
ref = pair_sets[METHOD_ORDER[0]]
for m in METHOD_ORDER[1:]:
    assert pair_sets[m] == ref, f'{m} covers a different (trait, drug) set'
lbl = predictions.groupby(['trait', 'drug'], observed=True)['true_class'].nunique()
assert (lbl == 1).all(), 'true_class disagrees across methods for some (trait, drug)'

# Pivot to a shared (trait, drug) index; y_true is identical across methods.
wide = predictions.pivot_table(
    index=['trait', 'drug'], columns='method', values='score', observed=True)[METHOD_ORDER]
y_true = predictions.pivot_table(
    index=['trait', 'drug'], columns='method', values='true_class',
    observed=True).iloc[:, 0].values.astype(int)
scores = {m: wide[m].values for m in METHOD_ORDER}

n = len(y_true)
n_pos = int(y_true.sum())
n_neg = n - n_pos
base_rate = n_pos / n
EF_MAX = 1.0 / base_rate
display(f'{n} pooled pairs: {n_pos} positive / {n_neg} negative '
        f'(base rate {base_rate:.4f}); EF ceiling = 1/base_rate = {EF_MAX:.4f}')

(2740, 5)

,trait,drug,method,score,true_class
0,DOID:0050741,DB00215,gene_based,316134.3,1.0
1,DOID:0050741,DB00215,module_based_archs4,324870.1,1.0
2,DOID:0050741,DB00215,module_based_gtex,349350.5,1.0
3,DOID:0050741,DB00215,module_based_recount2,398655.9,1.0
4,DOID:0050741,DB00704,gene_based,387103.6,1.0


'685 pooled pairs: 531 positive / 154 negative (base rate 0.7752); EF ceiling = 1/base_rate = 1.2900'

# Observed pooled early-enrichment metrics

For each method, compute the pooled BEDROC, the three EFs, and the three precision@k on the full 685-pair list. We record the integer cut (α / K / k) and the null / ceiling for transparency — at 77.5% prevalence the headroom is small (BEDROC/precision null = base rate; EF null = 1, ceiling = 1.29).

In [6]:
# Pooled observed early-enrichment metric per method.
rows = []
for m in METHOD_ORDER:
    s = scores[m]
    rows.append({'metric': 'BEDROC', 'method': m,
                 'value': bedroc_score(y_true, s, BEDROC_ALPHA),
                 'cut_type': 'alpha', 'cut': BEDROC_ALPHA, 'null': base_rate, 'ceiling': 1.0})
    for chi in EF_CHIS:
        K = int(np.ceil(chi * n))
        rows.append({'metric': f'EF@{int(round(chi * 100))}%', 'method': m,
                     'value': enrichment_factor(y_true, s, chi),
                     'cut_type': 'K', 'cut': K, 'null': 1.0, 'ceiling': EF_MAX})
    for k in PK_KS:
        rows.append({'metric': f'P@{k}', 'method': m,
                     'value': precision_at_k(y_true, s, k),
                     'cut_type': 'k', 'cut': k, 'null': base_rate, 'ceiling': 1.0})

observed = pd.DataFrame(rows)
observed['method'] = pd.Categorical(observed['method'], categories=METHOD_ORDER, ordered=True)
observed['metric'] = pd.Categorical(observed['metric'], categories=METRIC_ORDER, ordered=True)
observed = observed.sort_values(['metric', 'method']).reset_index(drop=True)

# Sanity: BEDROC null ~ base rate; EF never exceeds the ceiling.
print(f'base rate (BEDROC & P@k null) = {base_rate:.4f}; EF null = 1; EF ceiling = {EF_MAX:.4f}')
assert (observed.loc[observed['metric'].astype(str).str.startswith('EF'), 'value']
        <= EF_MAX + 1e-9).all(), 'EF exceeds its 1/base_rate ceiling -- bug'

# Headline view: metric x method wide table.
display(observed.pivot_table(index='metric', columns='method', values='value', observed=True)
        .reindex(METRIC_ORDER).round(4))

base rate (BEDROC & P@k null) = 0.7752; EF null = 1; EF ceiling = 1.2900


method,gene_based,module_based_archs4,module_based_gtex,module_based_recount2
metric,,,,
BEDROC,0.9438,0.9016,0.8895,0.9101
EF@1%,1.2900,1.2900,1.2900,1.2900
EF@5%,1.2163,1.2163,1.1794,1.1794
EF@10%,1.2152,1.1591,1.1405,1.1778
P@10,1.0000,1.0000,0.9000,1.0000
P@20,0.9500,0.9000,0.8500,0.9500
P@50,0.9600,0.9000,0.9200,0.9000


# Save

In [7]:
out_csv = OUTPUT_DIR / 'early_enrichment_observed.csv'
observed.to_csv(out_csv, index=False)
display(out_csv)
display(observed.shape)

PosixPath('/home/miltondp/projects/clamp/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/early_enrichment_test/early_enrichment_observed.csv')

(28, 7)